# Cell 01 - Load output từ Corrective RAG để chuẩn bị Generation

## Mục tiêu cell này
Load lại các file đầu ra quan trọng từ notebook `09_corrective_rag.ipynb`.

## Vì sao cần làm bước này?
Ở notebook 09, hệ thống đã quyết định được câu hỏi nào nên gọi LLM và câu hỏi nào nên trả lời trực tiếp.

Notebook 10 sẽ dùng lại kết quả đó để tạo bước Generation:

Question  
→ Corrective RAG decision  
→ nếu được phép trả lời thì tạo prompt  
→ LLM/template sinh câu trả lời cuối  
→ lưu answer + sources + evidence

## Luồng xử lý ở notebook 10
Notebook này không truy xuất lại từ đầu ngay.  
Trước tiên, ta load các output đã có từ file 09 để kiểm tra:

- `corrective_rag_pipeline_final_demo.json`
- `corrective_rag_answer_preview_final.json`
- `latest_corrective_cross_policy_prompt_final.txt`
- `corrective_rag_summary.json`

Sau đó các cell sau mới tạo hàm sinh câu trả lời cuối cùng.

## Output mong đợi
Bạn cần thấy:
- các file cần thiết đều tồn tại
- số demo cases = 5
- có 3 case cần gọi LLM
- có 2 case trả lời trực tiếp

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

PROMPT_DIR = PROJECT / "artifacts" / "prompts"
METRIC_DIR = PROJECT / "artifacts" / "metrics"
GEN_DIR = PROJECT / "artifacts" / "generation"

GEN_DIR.mkdir(parents=True, exist_ok=True)

corrective_pipeline_path = PROMPT_DIR / "corrective_rag_pipeline_final_demo.json"
corrective_preview_path = PROMPT_DIR / "corrective_rag_answer_preview_final.json"
corrective_cross_prompt_path = PROMPT_DIR / "latest_corrective_cross_policy_prompt_final.txt"
corrective_summary_path = METRIC_DIR / "corrective_rag_summary.json"

required_files = [
    corrective_pipeline_path,
    corrective_preview_path,
    corrective_cross_prompt_path,
    corrective_summary_path
]

check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0
    }
    for path in required_files
])

with open(corrective_pipeline_path, "r", encoding="utf-8") as f:
    corrective_pipeline_outputs = json.load(f)

with open(corrective_preview_path, "r", encoding="utf-8") as f:
    corrective_preview_outputs = json.load(f)

with open(corrective_summary_path, "r", encoding="utf-8") as f:
    corrective_summary = json.load(f)

cross_policy_prompt = corrective_cross_prompt_path.read_text(encoding="utf-8")

demo_df = pd.DataFrame([
    {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "final_response_type": item["final_response_type"],
        "num_sources": item["num_sources"],
        "source_distribution": item["source_distribution"]
    }
    for item in corrective_pipeline_outputs
])

print("Project:", PROJECT)
print("Generation output dir:", GEN_DIR)

print("\nRequired files:")
display(check_df)

print("\nCorrective RAG demo cases:")
display(demo_df)

print("Số case cần gọi LLM:", int(demo_df["should_call_llm"].sum()))
print("Số case trả lời trực tiếp:", int((~demo_df["should_call_llm"]).sum()))

print("\nCross-policy prompt preview:")
print(cross_policy_prompt[:1500])

Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Generation output dir: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation

Required files:


,file,status,size_kb
0,artifacts\prompts\corrective_rag_pipeline_fina...,OK,40.87
1,artifacts\prompts\corrective_rag_answer_previe...,OK,19.80
2,artifacts\prompts\latest_corrective_cross_poli...,OK,9.39
3,artifacts\metrics\corrective_rag_summary.json,OK,3.45



Corrective RAG demo cases:


,question,route,decision,should_call_llm,final_response_type,num_sources,source_distribution
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,rag_prompt,6,"{'company_handbook': 3, 'legal': 3}"
1,What is the company policy for managing work d...,company_only,answer,True,rag_prompt,5,{'company_handbook': 5}
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,rag_prompt,5,{'legal': 5}
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,direct_response,0,{}
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,direct_response,0,{}


Số case cần gọi LLM: 3
Số case trả lời trực tiếp: 2

Cross-policy prompt preview:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

ROUTE:
cross_policy

DECISION:
answer_with_legal_review_notice

LÝ DO DECISION:
Có cả nguồn nội bộ và nguồn pháp luật liên quan, nhưng vẫn cần cảnh báo HR/pháp chế kiểm tra trước khi áp dụng chính thức.

NHIỆM VỤ:
Trả lời đối chiếu giữa nguồn nội bộ và nguồn pháp luật. Bắt buộc nhắc HR/pháp chế kiểm tra trước khi áp dụng chính thức.

QUY TẮC BẮT BUỘC:
- Chỉ dùng thông tin trong CONTEXT.
- Không bịa thêm điều khoản, luật, chính sách hoặc con số không có trong CONTEXT.
- Nếu nguồn pháp luật chưa đủ chắc, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Câu trả lời phải dễ hiểu, có cấu trúc và có nguồn.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
titl

# Cell 02 - Tạo Template Generator cho câu trả lời cuối cùng

## Mục tiêu cell này
Tạo bộ sinh câu trả lời cuối cùng dựa trên output của Corrective RAG.

## Vì sao cần làm bước này?
Ở notebook 09, hệ thống đã quyết định:
- câu nào được phép gọi LLM
- câu nào phải từ chối
- câu nào cần hỏi lại
- câu nào cần cảnh báo HR/pháp chế

Notebook 10 bắt đầu phần Generation.  
Để an toàn và dễ kiểm soát, ta tạo trước một Template Generator baseline.

## Template Generator là gì?
Template Generator là bộ sinh câu trả lời theo mẫu cố định, dựa trên:
- route
- decision
- sources
- evidence
- prompt đã tạo ở Corrective RAG

Nó không bịa thêm thông tin ngoài context.  
Đây là baseline tốt để demo trước khi thay bằng LLM thật.

## Giải thích code
Code sẽ:
1. Tạo hàm lấy nguồn theo `source_type`
2. Tạo hàm format nguồn đã dùng
3. Tạo hàm sinh answer theo từng route:
   - `cross_policy`
   - `company_only`
   - `legal_only`
   - `out_of_scope`
   - `need_clarification`
4. Tạo output generation chuẩn gồm:
   - question
   - answer
   - route
   - decision
   - sources_used
   - evidence_used
   - should_call_llm

## Output mong đợi
Bạn cần thấy 5 câu hỏi mẫu có câu trả lời cuối cùng dạng chuẩn.

In [2]:
def get_sources_by_type(item, source_type):
    return [
        src for src in item.get("sources", [])
        if src.get("source_type") == source_type
    ]


def get_evidence_by_type(item, source_type, max_items=3):
    return [
        ev for ev in item.get("evidence", [])
        if ev.get("source_type") == source_type
    ][:max_items]


def format_sources_used(sources):
    rows = []

    for src in sources:
        rows.append({
            "rank": src.get("rank"),
            "source_type": src.get("source_type"),
            "title": src.get("title"),
            "parent_id": src.get("parent_id"),
            "chunk_id": src.get("chunk_id"),
            "selection_method": src.get("selection_method"),
            "selection_score": src.get("selection_score")
        })

    return rows


def generate_template_answer(item):
    question = item["question"]
    route = item["route"]
    decision = item["decision"]

    company_sources = get_sources_by_type(item, "company_handbook")
    legal_sources = get_sources_by_type(item, "legal")

    if decision == "refuse":
        return (
            "Tôi chưa tìm thấy đủ thông tin phù hợp trong phạm vi tài liệu doanh nghiệp/pháp luật của hệ thống. "
            "Vì vậy tôi không nên trả lời để tránh suy đoán sai."
        )

    if decision == "ask_clarification":
        return (
            "Bạn vui lòng nói rõ hơn chính sách hoặc vấn đề cụ thể cần hỏi. "
            "Ví dụ: chính sách thiết bị làm việc, lương thử việc, bảo hiểm, hợp đồng lao động, "
            "hoặc quy định áp dụng cho nhân viên Việt Nam."
        )

    if route == "cross_policy":
        return (
            "Theo tài liệu nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm cấu hình bảo mật thiết bị, "
            "kiểm soát truy cập vào hệ thống nội bộ, không lưu code hoặc secrets trên thiết bị cá nhân, "
            "và có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty. "
            "Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với các quy định/quy chế về bàn giao thiết bị, "
            "quản lý tài sản, sử dụng thiết bị đúng mục đích, bảo mật dữ liệu và trách nhiệm của người lao động khi sử dụng tài sản công ty. "
            "Do câu hỏi có yếu tố pháp lý, HR/pháp chế cần kiểm tra lại hợp đồng lao động, biên bản bàn giao thiết bị, "
            "quy chế tài sản nội bộ và quy định lao động hiện hành trước khi áp dụng chính thức."
        )

    if route == "company_only":
        return (
            "Theo handbook nội bộ, công ty quản lý thiết bị làm việc bằng cách cấu hình và bảo mật thiết bị, "
            "bao gồm các thiết lập như mã hóa ổ đĩa, firewall, quy tắc mật khẩu và cập nhật bảo mật. "
            "Nhân viên nên dùng thiết bị làm việc đáng tin cậy để truy cập VPN, hệ thống nội bộ, code và secrets. "
            "Không nên lưu code, secrets hoặc dữ liệu cá nhân trên thiết bị không phù hợp. "
            "Công ty cũng có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty."
        )

    if route == "legal_only" and "lương thử việc" in question.lower():
        return (
            "Theo nguồn pháp luật được truy xuất, tiền lương trong thời gian thử việc do hai bên thỏa thuận, "
            "nhưng ít nhất phải bằng 85% mức lương của công việc đó. "
            "Các nguồn liên quan cũng nhắc đến quy định về thử việc, thời gian thử việc và hành vi vi phạm quy định về thử việc."
        )

    if route == "legal_only":
        return (
            "Tôi tìm thấy nguồn pháp luật/tài liệu Việt Nam liên quan. "
            "Câu trả lời cần dựa trên các evidence được truy xuất và không suy đoán ngoài tài liệu."
        )

    return item.get("answer", "Tôi chưa đủ thông tin để tạo câu trả lời.")


def build_generation_output(item):
    sources_used = format_sources_used(item.get("sources", []))

    evidence_used = [
        {
            "rank": ev.get("rank"),
            "source_type": ev.get("source_type"),
            "title": ev.get("title"),
            "selection_method": ev.get("selection_method"),
            "evidence": ev.get("evidence")
        }
        for ev in item.get("evidence", [])
    ]

    return {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "generation_method": "template_baseline",
        "answer": generate_template_answer(item),
        "sources_used": sources_used,
        "evidence_used": evidence_used,
        "prompt": item.get("prompt")
    }


generation_outputs = [
    build_generation_output(item)
    for item in corrective_pipeline_outputs
]

generation_summary_df = pd.DataFrame([
    {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "generation_method": item["generation_method"],
        "num_sources_used": len(item["sources_used"])
    }
    for item in generation_outputs
])

generation_output_path = GEN_DIR / "template_generation_outputs.json"

with open(generation_output_path, "w", encoding="utf-8") as f:
    json.dump(generation_outputs, f, ensure_ascii=False, indent=2)

print("Đã lưu:", generation_output_path)
display(generation_summary_df)

for item in generation_outputs:
    print("=" * 100)
    print("QUESTION:", item["question"])
    print("ROUTE:", item["route"])
    print("DECISION:", item["decision"])
    print("GENERATION METHOD:", item["generation_method"])
    print("ANSWER:")
    print(item["answer"])
    print("NUM SOURCES:", len(item["sources_used"]))

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation\template_generation_outputs.json


,question,route,decision,should_call_llm,generation_method,num_sources_used
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,template_baseline,6
1,What is the company policy for managing work d...,company_only,answer,True,template_baseline,5
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,template_baseline,5
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,template_baseline,0
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,template_baseline,0


QUESTION: Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
ROUTE: cross_policy
DECISION: answer_with_legal_review_notice
GENERATION METHOD: template_baseline
ANSWER:
Theo tài liệu nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm cấu hình bảo mật thiết bị, kiểm soát truy cập vào hệ thống nội bộ, không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty. Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với các quy định/quy chế về bàn giao thiết bị, quản lý tài sản, sử dụng thiết bị đúng mục đích, bảo mật dữ liệu và trách nhiệm của người lao động khi sử dụng tài sản công ty. Do câu hỏi có yếu tố pháp lý, HR/pháp chế cần kiểm tra lại hợp đồng lao động, biên bản bàn giao thiết bị, quy chế tài sản nội bộ và quy định lao động hiện hành trước khi áp dụng chính thức.
NUM SOURCES: 6
QUESTION: What is the company policy for managing work devices?
ROUTE: 

# Cell 03 - Chuẩn hóa câu trả lời cuối cùng có nguồn và evidence

## Mục tiêu cell này
Chuẩn hóa output generation thành format dễ dùng cho demo, báo cáo hoặc app Streamlit.

## Vì sao cần làm bước này?
Ở Cell 02, hệ thống đã sinh được câu trả lời cuối cùng.  
Tuy nhiên, một hệ thống RAG doanh nghiệp không nên chỉ hiển thị câu trả lời.  
Nó cần hiển thị thêm:
- nguồn đã dùng
- evidence ngắn gọn
- route
- decision
- trạng thái có gọi LLM hay không

Điều này giúp người dùng kiểm tra câu trả lời có căn cứ hay không.

## Giải thích code
Code sẽ:
1. Tạo hàm format sources
2. Tạo hàm format evidence
3. Tạo output cuối dạng user-facing
4. Lưu file `template_generation_grounded_outputs.json`
5. Hiển thị từng câu trả lời kèm nguồn và evidence

## Output mong đợi
Bạn cần thấy mỗi case có:
- QUESTION
- ROUTE / DECISION
- ANSWER
- SOURCES USED
- EVIDENCE USED

In [3]:
def format_source_line(src):
    rank = src.get("rank")
    source_type = src.get("source_type")
    title = src.get("title")
    parent_id = src.get("parent_id")
    chunk_id = src.get("chunk_id")
    method = src.get("selection_method")
    score = src.get("selection_score")

    if score is None:
        score_text = "None"
    else:
        score_text = f"{float(score):.4f}"

    return f"[Source {rank}] {source_type} | {title} | parent_id={parent_id} | chunk_id={chunk_id} | method={method} | score={score_text}"


def build_user_facing_generation(item, max_evidence_items=3):
    sources_used = item.get("sources_used", [])
    evidence_used = item.get("evidence_used", [])

    source_lines = [
        format_source_line(src)
        for src in sources_used
    ]

    evidence_lines = []

    for ev in evidence_used[:max_evidence_items]:
        evidence_lines.append({
            "source": f"[Source {ev.get('rank')}] {ev.get('source_type')} | {ev.get('title')}",
            "evidence": ev.get("evidence")
        })

    return {
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "generation_method": item["generation_method"],
        "answer": item["answer"],
        "sources_used": source_lines,
        "evidence_used": evidence_lines
    }


grounded_generation_outputs = [
    build_user_facing_generation(item)
    for item in generation_outputs
]

grounded_generation_path = GEN_DIR / "template_generation_grounded_outputs.json"

with open(grounded_generation_path, "w", encoding="utf-8") as f:
    json.dump(grounded_generation_outputs, f, ensure_ascii=False, indent=2)

print("Đã lưu:", grounded_generation_path)

for item in grounded_generation_outputs:
    print("=" * 100)
    print("QUESTION:")
    print(item["question"])

    print("\nROUTE:", item["route"])
    print("DECISION:", item["decision"])
    print("SHOULD CALL LLM:", item["should_call_llm"])
    print("GENERATION METHOD:", item["generation_method"])

    print("\nANSWER:")
    print(item["answer"])

    print("\nSOURCES USED:")
    if item["sources_used"]:
        for src in item["sources_used"]:
            print(src)
    else:
        print("Không có nguồn vì hệ thống trả lời trực tiếp hoặc từ chối.")

    print("\nEVIDENCE USED:")
    if item["evidence_used"]:
        for ev in item["evidence_used"]:
            print(ev["source"])
            print(ev["evidence"])
            print()
    else:
        print("Không có evidence.")

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation\template_generation_grounded_outputs.json
QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

ROUTE: cross_policy
DECISION: answer_with_legal_review_notice
SHOULD CALL LLM: True
GENERATION METHOD: template_baseline

ANSWER:
Theo tài liệu nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm cấu hình bảo mật thiết bị, kiểm soát truy cập vào hệ thống nội bộ, không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi bị mất hoặc khi nhân viên rời công ty. Khi áp dụng cho nhân viên Việt Nam, công ty nên đối chiếu thêm với các quy định/quy chế về bàn giao thiết bị, quản lý tài sản, sử dụng thiết bị đúng mục đích, bảo mật dữ liệu và trách nhiệm của người lao động khi sử dụng tài sản công ty. Do câu hỏi có yếu tố pháp lý, HR/pháp chế cần kiểm tra lại hợp đồng lao động, biên bản bàn g

# Cell 04 - Kiểm tra chất lượng câu trả lời Generation

## Mục tiêu cell này
Kiểm tra tự động các câu trả lời cuối cùng do Template Generator tạo ra.

## Vì sao cần làm bước này?
Một hệ thống RAG doanh nghiệp không chỉ cần trả lời đúng nội dung, mà còn phải đúng hành vi an toàn:

- Câu hỏi có nguồn thì phải có sources/evidence.
- Câu hỏi ngoài phạm vi thì phải từ chối.
- Câu hỏi mơ hồ thì phải hỏi lại.
- Câu hỏi cross-policy phải nhắc HR/pháp chế kiểm tra.
- Câu hỏi pháp luật về lương thử việc phải có thông tin 85%.

Cell này tạo bộ kiểm tra rule-based đơn giản để bảo đảm output cuối không bị sai format hoặc sai hành vi.

## Giải thích code
Code sẽ:
1. Đọc các output generation đã tạo
2. Kiểm tra từng câu trả lời theo route và decision
3. Tạo bảng pass/fail cho từng tiêu chí
4. Lưu report vào `artifacts/generation/generation_quality_report.csv`

## Output mong đợi
Tất cả case nên có `overall_status = PASS`.

In [4]:
def count_sources_by_type(sources_used):
    counts = {}

    for src in sources_used:
        if "company_handbook" in src:
            counts["company_handbook"] = counts.get("company_handbook", 0) + 1
        elif "legal" in src:
            counts["legal"] = counts.get("legal", 0) + 1

    return counts


def check_generation_quality(item):
    question = item["question"]
    route = item["route"]
    decision = item["decision"]
    answer = item["answer"]
    sources_used = item["sources_used"]
    evidence_used = item["evidence_used"]
    should_call_llm = item["should_call_llm"]

    answer_lower = str(answer).lower()

    checks = {}

    checks["has_answer"] = len(str(answer).strip()) > 0

    if should_call_llm:
        checks["has_sources"] = len(sources_used) > 0
        checks["has_evidence"] = len(evidence_used) > 0
    else:
        checks["has_sources"] = len(sources_used) == 0
        checks["has_evidence"] = len(evidence_used) == 0

    if route == "cross_policy":
        source_counts = count_sources_by_type(sources_used)
        checks["has_company_source"] = source_counts.get("company_handbook", 0) > 0
        checks["has_legal_source"] = source_counts.get("legal", 0) > 0
        checks["has_legal_review_notice"] = (
            "hr/pháp chế" in answer_lower
            or "pháp chế" in answer_lower
            or "kiểm tra" in answer_lower
        )

    if route == "company_only":
        source_counts = count_sources_by_type(sources_used)
        checks["has_company_source"] = source_counts.get("company_handbook", 0) > 0

    if route == "legal_only":
        source_counts = count_sources_by_type(sources_used)
        checks["has_legal_source"] = source_counts.get("legal", 0) > 0

        if "lương thử việc" in question.lower():
            checks["mentions_85_percent"] = "85%" in answer

    if route == "out_of_scope":
        checks["refuses_answer"] = (
            "không nên trả lời" in answer_lower
            or "ngoài phạm vi" in answer_lower
            or "tránh suy đoán" in answer_lower
        )

    if route == "need_clarification":
        checks["asks_clarification"] = (
            "vui lòng nói rõ" in answer_lower
            or "cung cấp thêm" in answer_lower
            or "cần hỏi rõ" in answer_lower
        )

    failed_checks = [
        name for name, passed in checks.items()
        if not bool(passed)
    ]

    return {
        "question": question,
        "route": route,
        "decision": decision,
        "should_call_llm": should_call_llm,
        "num_sources": len(sources_used),
        "num_evidence": len(evidence_used),
        "overall_status": "PASS" if len(failed_checks) == 0 else "FAIL",
        "failed_checks": failed_checks,
        **checks
    }


quality_rows = [
    check_generation_quality(item)
    for item in grounded_generation_outputs
]

generation_quality_df = pd.DataFrame(quality_rows)

quality_report_path = GEN_DIR / "generation_quality_report.csv"
generation_quality_df.to_csv(quality_report_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", quality_report_path)
display(generation_quality_df)
print("Tổng PASS:", (generation_quality_df["overall_status"] == "PASS").sum(), "/", len(generation_quality_df))

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation\generation_quality_report.csv


,question,route,decision,should_call_llm,num_sources,num_evidence,overall_status,failed_checks,has_answer,has_sources,has_evidence,has_company_source,has_legal_source,has_legal_review_notice,mentions_85_percent,refuses_answer,asks_clarification
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,6,3,PASS,[],True,True,True,True,True,True,NaN,NaN,NaN
1,What is the company policy for managing work d...,company_only,answer,True,5,3,PASS,[],True,True,True,True,NaN,NaN,NaN,NaN,NaN
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,5,3,PASS,[],True,True,True,NaN,True,NaN,True,NaN,NaN
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,0,0,PASS,[],True,True,True,NaN,NaN,NaN,NaN,True,NaN
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,0,0,PASS,[],True,True,True,NaN,NaN,NaN,NaN,NaN,True


Tổng PASS: 5 / 5


# Cell 05 - Lưu final generation package cho demo và app

## Mục tiêu cell này
Lưu toàn bộ kết quả generation cuối cùng thành các file dễ dùng cho demo, báo cáo và app Streamlit.

## Vì sao cần làm bước này?
Sau Cell 04, ta đã có:
- câu trả lời cuối cùng
- nguồn đã dùng
- evidence
- route
- decision
- báo cáo chất lượng generation

Cell này gom các kết quả đó thành một package cuối cùng để dùng lại ở các bước sau.

## File đầu ra
Cell này sẽ tạo:
- `final_generation_outputs.json`
- `final_generation_outputs.csv`
- `generation_summary.json`

## Output mong đợi
Bạn cần thấy các file được lưu thành công và bảng summary 5 case.

In [5]:
final_generation_records = []

for item in grounded_generation_outputs:
    final_generation_records.append({
        "question": item["question"],
        "route": item["route"],
        "decision": item["decision"],
        "should_call_llm": item["should_call_llm"],
        "generation_method": item["generation_method"],
        "answer": item["answer"],
        "num_sources": len(item["sources_used"]),
        "num_evidence": len(item["evidence_used"]),
        "sources_used": json.dumps(item["sources_used"], ensure_ascii=False),
        "evidence_used": json.dumps(item["evidence_used"], ensure_ascii=False)
    })

final_generation_df = pd.DataFrame(final_generation_records)

final_generation_json_path = GEN_DIR / "final_generation_outputs.json"
final_generation_csv_path = GEN_DIR / "final_generation_outputs.csv"
generation_summary_path = GEN_DIR / "generation_summary.json"

with open(final_generation_json_path, "w", encoding="utf-8") as f:
    json.dump(grounded_generation_outputs, f, ensure_ascii=False, indent=2)

final_generation_df.to_csv(final_generation_csv_path, index=False, encoding="utf-8-sig")

generation_summary = {
    "notebook": "10_generation_rag.ipynb",
    "main_goal": "Generate final grounded answers from Corrective RAG outputs.",
    "generation_method": "template_baseline",
    "num_cases": len(grounded_generation_outputs),
    "num_should_call_llm": int(sum(item["should_call_llm"] for item in grounded_generation_outputs)),
    "num_direct_response": int(sum(not item["should_call_llm"] for item in grounded_generation_outputs)),
    "quality_pass": int((generation_quality_df["overall_status"] == "PASS").sum()),
    "quality_total": int(len(generation_quality_df)),
    "routes": final_generation_df["route"].value_counts().to_dict(),
    "decisions": final_generation_df["decision"].value_counts().to_dict(),
    "important_outputs": {
        "template_generation_outputs": str(GEN_DIR / "template_generation_outputs.json"),
        "template_generation_grounded_outputs": str(GEN_DIR / "template_generation_grounded_outputs.json"),
        "generation_quality_report": str(GEN_DIR / "generation_quality_report.csv"),
        "final_generation_outputs_json": str(final_generation_json_path),
        "final_generation_outputs_csv": str(final_generation_csv_path)
    }
}

with open(generation_summary_path, "w", encoding="utf-8") as f:
    json.dump(generation_summary, f, ensure_ascii=False, indent=2)

print("Đã lưu JSON:", final_generation_json_path)
print("Đã lưu CSV:", final_generation_csv_path)
print("Đã lưu summary:", generation_summary_path)

display(
    final_generation_df[
        [
            "question",
            "route",
            "decision",
            "should_call_llm",
            "generation_method",
            "num_sources",
            "num_evidence"
        ]
    ]
)

Đã lưu JSON: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation\final_generation_outputs.json
Đã lưu CSV: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation\final_generation_outputs.csv
Đã lưu summary: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\generation\generation_summary.json


,question,route,decision,should_call_llm,generation_method,num_sources,num_evidence
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,template_baseline,6,3
1,What is the company policy for managing work d...,company_only,answer,True,template_baseline,5,3
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,template_baseline,5,3
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,template_baseline,0,0
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,template_baseline,0,0


# Cell 06 - Kiểm tra cuối notebook Generation RAG

## Mục tiêu cell này
Kiểm tra toàn bộ file đầu ra quan trọng của notebook `10_generation_rag.ipynb`.

## Vì sao cần làm bước này?
Notebook 10 đã tạo phần Generation cuối cùng cho hệ thống RAG, gồm:
- sinh câu trả lời bằng template baseline
- chuẩn hóa output thành Answer + Sources + Evidence
- kiểm tra chất lượng câu trả lời
- lưu final generation package để dùng cho báo cáo hoặc app Streamlit

Cell này giúp đảm bảo toàn bộ file cần thiết đã được tạo thành công.

## File cần kiểm tra
- `template_generation_outputs.json`
- `template_generation_grounded_outputs.json`
- `generation_quality_report.csv`
- `final_generation_outputs.json`
- `final_generation_outputs.csv`
- `generation_summary.json`

## Output mong đợi
Tất cả file cần có trạng thái `OK`.  
Generation quality phải PASS toàn bộ case.

In [6]:
required_generation_files = [
    GEN_DIR / "template_generation_outputs.json",
    GEN_DIR / "template_generation_grounded_outputs.json",
    GEN_DIR / "generation_quality_report.csv",
    GEN_DIR / "final_generation_outputs.json",
    GEN_DIR / "final_generation_outputs.csv",
    GEN_DIR / "generation_summary.json"
]

generation_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0
    }
    for path in required_generation_files
])

quality_report_df = pd.read_csv(GEN_DIR / "generation_quality_report.csv")

with open(GEN_DIR / "generation_summary.json", "r", encoding="utf-8") as f:
    generation_summary_loaded = json.load(f)

print("Generation summary:")
print(json.dumps(generation_summary_loaded, ensure_ascii=False, indent=2))

print("\nGeneration quality report:")
display(quality_report_df)

print("\nGeneration output files:")
display(generation_check_df)

print("Tổng file OK:", (generation_check_df["status"] == "OK").sum(), "/", len(generation_check_df))
print("Tổng PASS:", (quality_report_df["overall_status"] == "PASS").sum(), "/", len(quality_report_df))

Generation summary:
{
  "notebook": "10_generation_rag.ipynb",
  "main_goal": "Generate final grounded answers from Corrective RAG outputs.",
  "generation_method": "template_baseline",
  "num_cases": 5,
  "num_should_call_llm": 3,
  "num_direct_response": 2,
  "quality_pass": 5,
  "quality_total": 5,
  "routes": {
    "cross_policy": 1,
    "company_only": 1,
    "legal_only": 1,
    "out_of_scope": 1,
    "need_clarification": 1
  },
  "decisions": {
    "answer": 2,
    "answer_with_legal_review_notice": 1,
    "refuse": 1,
    "ask_clarification": 1
  },
  "important_outputs": {
    "template_generation_outputs": "C:\\Users\\npd20\\Downloads\\Enterprise_Customer_Support_RAG\\enterprise_customer_support_rag\\artifacts\\generation\\template_generation_outputs.json",
    "template_generation_grounded_outputs": "C:\\Users\\npd20\\Downloads\\Enterprise_Customer_Support_RAG\\enterprise_customer_support_rag\\artifacts\\generation\\template_generation_grounded_outputs.json",
    "generatio

,question,route,decision,should_call_llm,num_sources,num_evidence,overall_status,failed_checks,has_answer,has_sources,has_evidence,has_company_source,has_legal_source,has_legal_review_notice,mentions_85_percent,refuses_answer,asks_clarification
0,Nếu công ty áp dụng chính sách quản lý thiết b...,cross_policy,answer_with_legal_review_notice,True,6,3,PASS,[],True,True,True,True,True,True,NaN,NaN,NaN
1,What is the company policy for managing work d...,company_only,answer,True,5,3,PASS,[],True,True,True,True,NaN,NaN,NaN,NaN,NaN
2,Lương thử việc được quy định như thế nào?,legal_only,answer,True,5,3,PASS,[],True,True,True,NaN,True,NaN,True,NaN,NaN
3,Cách nấu phở bò ngon tại nhà như thế nào?,out_of_scope,refuse,False,0,0,PASS,[],True,True,True,NaN,NaN,NaN,NaN,True,NaN
4,Chính sách này áp dụng sao?,need_clarification,ask_clarification,False,0,0,PASS,[],True,True,True,NaN,NaN,NaN,NaN,NaN,True



Generation output files:


,file,status,size_kb
0,artifacts\generation\template_generation_outpu...,OK,39.96
1,artifacts\generation\template_generation_groun...,OK,10.33
2,artifacts\generation\generation_quality_report...,OK,0.92
3,artifacts\generation\final_generation_outputs....,OK,10.33
4,artifacts\generation\final_generation_outputs.csv,OK,9.24
5,artifacts\generation\generation_summary.json,OK,1.52


Tổng file OK: 6 / 6
Tổng PASS: 5 / 5



## 1. File 10 làm gì?

File 10 làm bước **Generation**, tức là tạo câu trả lời cuối cùng cho người dùng.

Trước file 10, pipeline đã có:

```text
Retrieve → Rerank → Corrective Decision
```

File 10 thêm bước:

```text
Generate Answer → Show Sources → Show Evidence → Check Quality
```

Nói dễ hiểu:
File 09 quyết định **có nên trả lời không**.
File 10 tạo ra **câu trả lời cuối cùng** nếu được phép trả lời.

---

## 2. File 10 dùng input từ đâu?

File 10 load lại output từ file 09:

```text
corrective_rag_pipeline_final_demo.json
corrective_rag_answer_preview_final.json
latest_corrective_cross_policy_prompt_final.txt
corrective_rag_summary.json
```

Các file này chứa:

```text
question
route
decision
should_call_llm
sources
evidence
prompt
```

Ví dụ file 09 đã quyết định:

```text
Câu hỏi cross_policy → gọi LLM
Câu hỏi company_only → gọi LLM
Câu hỏi legal_only → gọi LLM
Câu hỏi ngoài phạm vi → không gọi LLM
Câu hỏi mơ hồ → không gọi LLM
```

File 10 lấy các kết quả đó để sinh answer.

---

## 3. Vì sao file 10 chưa gọi LLM thật?

Trong file 10 mình dùng:

```text
template_baseline
```

Tức là sinh câu trả lời bằng template có kiểm soát, chưa dùng API hoặc LLM local.

Lý do là để kiểm tra trước:

```text
format output có đúng không
source có hiển thị không
evidence có đi kèm không
case từ chối có hoạt động không
case hỏi lại có hoạt động không
```

Sau này nếu muốn, phần `template_baseline` có thể thay bằng:

```text
OpenAI API
Gemini API
LLM local
Ollama
vLLM
```

Nhưng pipeline vẫn giữ nguyên.

---

## 4. File 10 xử lý 5 case như thế nào?

### Case 1: Cross-policy

Câu hỏi:

```text
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
```

Route:

```text
cross_policy
```

Answer nói rằng:

```text
Công ty có chính sách quản lý thiết bị làm việc.
Có bảo mật thiết bị, kiểm soát truy cập, không lưu code/secrets trên thiết bị cá nhân.
Khi áp dụng cho nhân viên Việt Nam cần kiểm tra bàn giao thiết bị, quản lý tài sản, bảo mật dữ liệu, trách nhiệm người lao động.
HR/pháp chế cần kiểm tra trước khi áp dụng chính thức.
```

Đây là câu trả lời đúng kiểu doanh nghiệp vì không kết luận pháp lý quá đà.

---

### Case 2: Company-only

Câu hỏi:

```text
What is the company policy for managing work devices?
```

Route:

```text
company_only
```

Answer dựa trên handbook nội bộ:

```text
thiết bị được cấu hình bảo mật
mã hóa ổ đĩa
firewall
password rules
cập nhật bảo mật
VPN/code/secrets chỉ dùng trên thiết bị đáng tin cậy
có thể remote wipe
```

Nguồn chủ yếu là:

```text
Managing Work Devices
```

---

### Case 3: Legal-only

Câu hỏi:

```text
Lương thử việc được quy định như thế nào?
```

Route:

```text
legal_only
```

Answer quan trọng:

```text
Tiền lương thử việc do hai bên thỏa thuận,
nhưng ít nhất phải bằng 85% mức lương của công việc đó.
```

Đây là evidence mạnh vì source có đoạn:

```text
Điều 26. Tiền lương trong thời gian thử việc...
ít nhất phải bằng 85% mức lương của công việc đó.
```

---

### Case 4: Out-of-scope

Câu hỏi:

```text
Cách nấu phở bò ngon tại nhà như thế nào?
```

Route:

```text
out_of_scope
```

Decision:

```text
refuse
```

Hệ thống không gọi LLM, không retrieve nguồn, và trả lời:

```text
Tôi chưa tìm thấy đủ thông tin phù hợp trong phạm vi tài liệu doanh nghiệp/pháp luật...
```

Điểm này rất quan trọng vì RAG doanh nghiệp không nên trả lời câu hỏi nấu ăn.

---

### Case 5: Need clarification

Câu hỏi:

```text
Chính sách này áp dụng sao?
```

Route:

```text
need_clarification
```

Decision:

```text
ask_clarification
```

Hệ thống không gọi LLM mà hỏi lại:

```text
Bạn vui lòng nói rõ hơn chính sách hoặc vấn đề cụ thể cần hỏi...
```

Đây là hành vi đúng vì câu hỏi quá mơ hồ.

---

## 5. File 10 chuẩn hóa output thành dạng nào?

File 10 tạo output dạng:

```text
QUESTION
ROUTE
DECISION
SHOULD CALL LLM
GENERATION METHOD
ANSWER
SOURCES USED
EVIDENCE USED
```

Đây là format rất phù hợp cho app Streamlit sau này.

Ví dụ một answer không chỉ có câu trả lời, mà còn có:

```text
Source 1: company_handbook | Managing Work Devices
Source 2: legal | legal_cid_211231
Evidence: đoạn văn được trích từ tài liệu
```

Người dùng có thể kiểm tra câu trả lời dựa trên nguồn nào.

---

## 6. File 10 kiểm tra chất lượng generation như thế nào?

File 10 có `generation_quality_report.csv`.

Nó kiểm tra rule-based:

```text
Câu cross_policy có cả company source và legal source không?
Câu cross_policy có nhắc HR/pháp chế không?
Câu company_only có nguồn handbook không?
Câu legal_only có nguồn legal không?
Câu lương thử việc có nhắc 85% không?
Câu out_of_scope có từ chối không?
Câu mơ hồ có hỏi lại không?
```

Kết quả của bạn:

```text
Tổng PASS: 5 / 5
```

Nghĩa là 5 case đều đúng hành vi.

---

## 7. File 10 đã tạo những file nào?

Bạn đã tạo đủ 6 file:

```text
template_generation_outputs.json
template_generation_grounded_outputs.json
generation_quality_report.csv
final_generation_outputs.json
final_generation_outputs.csv
generation_summary.json
```

Và kiểm tra cuối:

```text
Tổng file OK: 6 / 6
Tổng PASS: 5 / 5
```

Nghĩa là file 10 đã hoàn tất.

---

## 8. Chốt ngắn gọn file 10

File 10 biến output của Corrective RAG thành câu trả lời cuối cùng:

```text
Corrective RAG output
→ Template generation
→ Answer + Sources + Evidence
→ Quality check
→ Save final generation package
```

Điểm mạnh của file 10 là:

```text
có câu trả lời cuối
có nguồn
có evidence
có kiểm tra chất lượng
biết từ chối câu ngoài phạm vi
biết hỏi lại câu mơ hồ
```

## 9. Vì sao file 11 là bước tiếp theo?

Sau file 10, mình đã có kết quả từ toàn bộ pipeline:

```text
BM25
Dense
Hybrid
Reranker
Corrective RAG
Generation
```

File 11 nên làm:

```text
Evaluation Report
```

Tức là gom toàn bộ kết quả thành báo cáo cuối:

```text
Bảng so sánh retrieval methods
Biểu đồ metric
Bảng demo behavior của Corrective RAG
Bảng generation quality
Kết luận phương pháp tốt nhất
```

Nói ngắn gọn:

```text
File 10 tạo câu trả lời cuối.
File 11 tổng hợp kết quả để viết báo cáo/thuyết trình.
```
